In [1]:
import json
import re

from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

# Posibles ubicaciones según desde dónde
# se esté ejecutando Jupyter.
rutas_modelo = [
    Path("models/interprete_sst_v2"),
    Path("notebooks/models/interprete_sst_v2"),
    Path("../models/interprete_sst_v2")
]

rutas_datos = [
    Path("data/SGSST_auditorias_sinteticas_5000.csv"),
    Path("../data/SGSST_auditorias_sinteticas_5000.csv")
]

ruta_modelo = next(
    (
        ruta for ruta in rutas_modelo
        if (ruta / "modelo.pth").exists()
        and (ruta / "configuracion.json").exists()
    ),
    None
)

ruta_datos = next(
    (ruta for ruta in rutas_datos if ruta.exists()),
    None
)

if ruta_modelo is None:
    raise FileNotFoundError(
        "No se encontró el modelo V2. "
        "Revisa la ubicación de models/interprete_sst_v2."
    )

if ruta_datos is None:
    raise FileNotFoundError(
        "No se encontró el CSV de auditorías."
    )

print("Dispositivo:", device)
print("Modelo:", ruta_modelo.resolve())
print("Datos:", ruta_datos.resolve())

d:\VIVIAN\Uis\Quinto semestre\Deep Learning\Proyecto Mini-Transforms\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dispositivo: cuda
Modelo: D:\VIVIAN\Uis\Quinto semestre\Deep Learning\Proyecto Mini-Transforms\notebooks\models\interprete_sst_v2
Datos: D:\VIVIAN\Uis\Quinto semestre\Deep Learning\Proyecto Mini-Transforms\data\SGSST_auditorias_sinteticas_5000.csv


In [2]:
# Cargar registros históricos
auditorias = pd.read_csv(ruta_datos)

# Cargar configuración del modelo
with open(
    ruta_modelo / "configuracion.json",
    "r",
    encoding="utf-8"
) as archivo:
    config = json.load(archivo)

print("REGISTROS HISTÓRICOS")
print("Filas:", len(auditorias))
print("Columnas:", len(auditorias.columns))

print("\nCOLUMNAS DISPONIBLES")
print(auditorias.columns.tolist())

print("\nCONFIGURACIÓN DEL MODELO")
print("Intenciones:", config["intenciones"])
print("Entidades:", config["entidades"])
print("Categorías:", len(config["categorias"]))
print("Estados:", config["estados"])

display(auditorias.head(5))

REGISTROS HISTÓRICOS
Filas: 5000
Columnas: 18

COLUMNAS DISPONIBLES
['id_hallazgo', 'id_auditoria', 'fecha_auditoria', 'empresa', 'area', 'tipo_auditoria', 'hallazgo', 'categoria_id', 'categoria', 'criterio_evaluacion', 'clasificacion', 'nivel_riesgo', 'estado', 'responsable_area', 'fecha_cierre', 'origen_dato', 'escenario_id', 'split']

CONFIGURACIÓN DEL MODELO
Intenciones: ['comparacion_anual', 'conteo', 'listado', 'ranking']
Entidades: ['auditorias', 'hallazgos']
Categorías: 12
Estados: ['Abierto', 'Cerrado', 'En seguimiento']


,id_hallazgo,id_auditoria,fecha_auditoria,empresa,area,tipo_auditoria,hallazgo,categoria_id,categoria,criterio_evaluacion,clasificacion,nivel_riesgo,estado,responsable_area,fecha_cierre,origen_dato,escenario_id,split
0,SST-04392,AUD-0879,2024-11-04,Empresa Andina SAS,Obras,Auditoria interna SG-SST,El equipo auditor documento la siguiente situa...,CONTRATISTAS,Gestion SST de contratistas,"Verificacion previa, coordinacion, seguimiento...",No conformidad,Alto,Abierto,Obras,NaN,SINTETICO,CONTRATISTAS-05,train
1,SST-03419,AUD-0684,2024-06-09,Constructora Atlas SAS,Talento humano,Auditoria interna SG-SST,La evidencia recopilada permite establecer que...,DOCUMENTAL,Control documental SG-SST,"Versiones, aprobaciones, trazabilidad, acceso ...",Observacion,Medio,Cerrado,Talento humano,2024-10-01,SINTETICO,DOCUMENTAL-04,test
2,SST-02966,AUD-0594,2025-12-28,Constructora Atlas SAS,Servicios generales,Auditoria interna SG-SST,Al revisar las condiciones de la planta de pro...,FORMACION,Formacion y competencias SST,"Inducciones, entrenamiento y evidencia de comp...",No conformidad,Alto,Cerrado,Servicios generales,2026-02-05,SINTETICO,FORMACION-07,train
3,SST-01228,AUD-0246,2024-06-26,Constructora Atlas SAS,Obras,Auditoria interna SG-SST,"Segun la inspeccion realizada, el pasillo de c...",LOCATIVOS,Condiciones locativas,"Orden, aseo, rutas, almacenamiento, pisos y se...",No conformidad,Alto,Abierto,Obras,NaN,SINTETICO,LOCATIVOS-06,train
4,SST-02313,AUD-0463,2024-12-13,Empresa Andina SAS,Logistica,Auditoria interna SG-SST,La muestra revisada revelo que un incidente si...,INCIDENTES,Investigacion de incidentes y accidentes,"Reporte, investigacion, causalidad y seguimien...",Observacion,Medio,Cerrado,Logistica,2025-02-15,SINTETICO,INCIDENTES-00,train


In [3]:
class InterpreteSST_V2(nn.Module):

    def __init__(self, configuracion):
        super().__init__()

        dimension = configuracion["dimension_embeddings"]

        self.compartida = nn.Sequential(
            nn.Linear(dimension, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU()
        )

        self.cabezas = nn.ModuleDict({
            "intencion": nn.Linear(
                64, len(configuracion["intenciones"])
            ),
            "entidad": nn.Linear(
                64, len(configuracion["entidades"])
            ),
            "hay_categoria": nn.Linear(64, 2),
            "categoria": nn.Linear(
                64, len(configuracion["categorias"])
            ),
            "hay_estado": nn.Linear(64, 2),
            "estado": nn.Linear(
                64, len(configuracion["estados"])
            )
        })

    def forward(self, x):
        representacion = self.compartida(x)

        return {
            nombre: cabeza(representacion)
            for nombre, cabeza in self.cabezas.items()
        }


# Recuperar nuestro modelo supervisado
interprete = InterpreteSST_V2(config).to(device)

pesos = torch.load(
    ruta_modelo / "modelo.pth",
    map_location=device,
    weights_only=True
)

interprete.load_state_dict(pesos)
interprete.eval()

# Recuperar MiniLM
modelo_nlp = SentenceTransformer(
    config["modelo_embeddings"],
    device=device
)

print("MiniLM cargado correctamente.")
print("Intérprete V2 recuperado correctamente.")

parametros = sum(
    p.numel() for p in interprete.parameters()
)

print("Parámetros del intérprete:", parametros)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2506.04it/s]


MiniLM cargado correctamente.
Intérprete V2 recuperado correctamente.
Parámetros del intérprete: 59161


In [4]:
def interpretar_consulta(pregunta):

    interprete.eval()

    embedding = modelo_nlp.encode(
        [pregunta],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    entrada = torch.tensor(
        embedding,
        dtype=torch.float32,
        device=device
    )

    with torch.inference_mode():
        salidas = interprete(entrada)

    indices = {}
    probabilidades = {}

    for campo, logits in salidas.items():

        probs = F.softmax(logits, dim=1)[0]

        indice = int(probs.argmax().item())

        indices[campo] = indice
        probabilidades[campo] = float(
            probs[indice].item()
        )

    intencion = config["intenciones"][
        indices["intencion"]
    ]

    entidad = config["entidades"][
        indices["entidad"]
    ]

    categoria = "NINGUNA"

    if indices["hay_categoria"] == 1:
        categoria = config["categorias"][
            indices["categoria"]
        ]

    estado = "NINGUNO"

    if indices["hay_estado"] == 1:
        estado = config["estados"][
            indices["estado"]
        ]

    # Extraer años explícitos
    anios = sorted(set(
        int(anio)
        for anio in re.findall(
            r"\b(?:19|20)\d{2}\b",
            pregunta
        )
    ))

    return {
        "pregunta": pregunta,
        "intencion": intencion,
        "entidad": entidad,
        "categoria": categoria,
        "estado": estado,
        "anios": anios,
        "probabilidades": probabilidades
    }


# Primera prueba
pregunta = "¿Cuántas auditorías se realizaron en 2025?"

resultado = interpretar_consulta(pregunta)

for clave, valor in resultado.items():
    if clave != "probabilidades":
        print(f"{clave}: {valor}")

print("\nProbabilidades del modelo:")
print(resultado["probabilidades"])

pregunta: ¿Cuántas auditorías se realizaron en 2025?
intencion: conteo
entidad: auditorias
categoria: NINGUNA
estado: NINGUNO
anios: [2025]

Probabilidades del modelo:
{'intencion': 0.9299039840698242, 'entidad': 0.5821711421012878, 'hay_categoria': 0.9855124354362488, 'categoria': 0.37815582752227783, 'hay_estado': 0.8348347544670105, 'estado': 0.8754298686981201}


In [5]:
# Crear una copia de trabajo
datos_sst = auditorias.copy()

# Verificar columnas indispensables
columnas_necesarias = [
    "id_hallazgo",
    "id_auditoria",
    "fecha_auditoria",
    "categoria",
    "estado",
    "hallazgo"
]

faltantes = [
    columna
    for columna in columnas_necesarias
    if columna not in datos_sst.columns
]

if faltantes:
    raise ValueError(
        f"Faltan columnas necesarias: {faltantes}"
    )

# Convertir fecha a formato datetime
datos_sst["fecha_auditoria"] = pd.to_datetime(
    datos_sst["fecha_auditoria"],
    errors="coerce"
)

# Crear columna auxiliar para consultas anuales
datos_sst["anio"] = (
    datos_sst["fecha_auditoria"].dt.year
)

print("BASE DE DATOS PREPARADA")
print("Hallazgos:", len(datos_sst))

print(
    "Auditorías únicas:",
    datos_sst["id_auditoria"].nunique()
)

print(
    "Fechas inválidas:",
    datos_sst["fecha_auditoria"].isna().sum()
)

print("\nRegistros por año:")
display(
    datos_sst["anio"]
    .value_counts()
    .sort_index()
    .rename_axis("anio")
    .reset_index(name="hallazgos")
)

BASE DE DATOS PREPARADA
Hallazgos: 5000
Auditorías únicas: 1000
Fechas inválidas: 0

Registros por año:


,anio,hallazgos
0,2024,1862
1,2025,1781
2,2026,1357


In [6]:
def preparar_consulta(interpretacion):

    intencion = interpretacion["intencion"]
    entidad = interpretacion["entidad"]
    categoria = interpretacion["categoria"]
    estado = interpretacion["estado"]
    anios = interpretacion["anios"]

    errores = []

    # Operaciones que implementaremos primero
    if intencion not in ["conteo", "listado"]:
        errores.append(
            f"La intención '{intencion}' todavía "
            "no está implementada."
        )

    if entidad not in ["hallazgos", "auditorias"]:
        errores.append(
            f"Entidad no reconocida: {entidad}"
        )

    # Validación de categorías
    categorias_validas = set(
        datos_sst["categoria"].dropna().unique()
    )

    if (
        categoria != "NINGUNA"
        and categoria not in categorias_validas
    ):
        errores.append(
            f"Categoría no disponible: {categoria}"
        )

    # Validación de estados
    estados_validos = set(
        datos_sst["estado"].dropna().unique()
    )

    if (
        estado != "NINGUNO"
        and estado not in estados_validos
    ):
        errores.append(
            f"Estado no disponible: {estado}"
        )

    # Por ahora, un año explícito como máximo
    if len(anios) > 1:
        errores.append(
            "La pregunta contiene varios años. "
            "Debemos interpretar primero si es "
            "una comparación o un intervalo."
        )

    # Los estados pertenecen a hallazgos,
    # no directamente a auditorías.
    if entidad == "auditorias" and estado != "NINGUNO":
        errores.append(
            "El estado corresponde a hallazgos. "
            "Debemos confirmar si se quieren contar "
            "auditorías con hallazgos en ese estado."
        )

    return {
        "interpretacion": interpretacion,
        "valida": len(errores) == 0,
        "errores": errores
    }


# Probar la validación
consulta_preparada = preparar_consulta(resultado)

print("Consulta válida:", consulta_preparada["valida"])
print("Errores:", consulta_preparada["errores"])

print("\nInterpretación:")
for campo in ["intencion", "entidad", "categoria", "estado", "anios"]:
    print(campo, ":", resultado[campo])

Consulta válida: True
Errores: []

Interpretación:
intencion : conteo
entidad : auditorias
categoria : NINGUNA
estado : NINGUNO
anios : [2025]


In [7]:
def ejecutar_consulta(consulta, confirmada=False):

    if not consulta["valida"]:
        return {
            "estado": "error",
            "mensaje": "La consulta requiere revisión.",
            "errores": consulta["errores"]
        }

    if not confirmada:
        return {
            "estado": "pendiente_confirmacion",
            "mensaje": (
                "Revisa la intención, entidad y filtros "
                "antes de ejecutar la consulta."
            ),
            "interpretacion": consulta["interpretacion"]
        }

    interpretacion = consulta["interpretacion"]

    intencion = interpretacion["intencion"]
    entidad = interpretacion["entidad"]
    categoria = interpretacion["categoria"]
    estado = interpretacion["estado"]
    anios = interpretacion["anios"]

    filtrados = datos_sst.copy()

    # Aplicar filtro anual
    if len(anios) == 1:
        filtrados = filtrados[
            filtrados["anio"] == anios[0]
        ]

    # Aplicar categoría
    if categoria != "NINGUNA":
        filtrados = filtrados[
            filtrados["categoria"] == categoria
        ]

    # Aplicar estado
    if estado != "NINGUNO":
        filtrados = filtrados[
            filtrados["estado"] == estado
        ]

    # CONTEOS
    if intencion == "conteo":

        if entidad == "hallazgos":
            cantidad = len(filtrados)

        elif entidad == "auditorias":
            cantidad = (
                filtrados["id_auditoria"].nunique()
            )

        return {
            "estado": "correcto",
            "entidad": entidad,
            "cantidad": int(cantidad),
            "filtros": {
                "categoria": categoria,
                "estado": estado,
                "anios": anios
            }
        }

    # LISTADOS
    if intencion == "listado":

        if entidad == "auditorias":
            columnas = [
                "id_auditoria",
                "fecha_auditoria",
                "empresa"
            ]

            tabla = (
                filtrados[columnas]
                .drop_duplicates(
                    subset=["id_auditoria"]
                )
            )

        else:
            columnas = [
                "id_hallazgo",
                "fecha_auditoria",
                "categoria",
                "estado",
                "hallazgo"
            ]

            tabla = filtrados[columnas]

        return {
            "estado": "correcto",
            "entidad": entidad,
            "cantidad": len(tabla),
            "datos": tabla.reset_index(drop=True),
            "filtros": {
                "categoria": categoria,
                "estado": estado,
                "anios": anios
            }
        }

    return {
        "estado": "error",
        "mensaje": "Intención no implementada."
    }

In [8]:
# Primero mostrar la interpretación sin ejecutar
revision = ejecutar_consulta(
    consulta_preparada,
    confirmada=False
)

print("Estado inicial:", revision["estado"])

# Confirmamos porque ya revisamos manualmente
# la interpretación de esta pregunta.
respuesta = ejecutar_consulta(
    consulta_preparada,
    confirmada=True
)

print("\nRESPUESTA DEL SISTEMA")
print("Estado:", respuesta["estado"])

if respuesta["estado"] == "correcto":
    print("Entidad:", respuesta["entidad"])
    print("Cantidad:", respuesta["cantidad"])
    print("Filtros:", respuesta["filtros"])
else:
    print(respuesta)

Estado inicial: pendiente_confirmacion

RESPUESTA DEL SISTEMA
Estado: correcto
Entidad: auditorias
Cantidad: 895
Filtros: {'categoria': 'NINGUNA', 'estado': 'NINGUNO', 'anios': [2025]}


In [9]:
# Revisar si un ID de auditoría aparece asociado
# a distintas empresas, fechas o años.

revision_ids = (
    datos_sst.groupby("id_auditoria")
    .agg(
        registros=("id_hallazgo", "count"),
        empresas=("empresa", "nunique"),
        fechas=("fecha_auditoria", "nunique"),
        anios=("anio", "nunique")
    )
    .reset_index()
)

print("Total de identificadores:", len(revision_ids))

print(
    "IDs asociados a varias empresas:",
    (revision_ids["empresas"] > 1).sum()
)

print(
    "IDs asociados a varias fechas:",
    (revision_ids["fechas"] > 1).sum()
)

print(
    "IDs presentes en varios años:",
    (revision_ids["anios"] > 1).sum()
)

print("\nEjemplos que requieren revisión:")

display(
    revision_ids.loc[
        (revision_ids["empresas"] > 1)
        | (revision_ids["anios"] > 1)
    ].head(10)
)

Total de identificadores: 1000
IDs asociados a varias empresas: 174
IDs asociados a varias fechas: 1000
IDs presentes en varios años: 983

Ejemplos que requieren revisión:


,id_auditoria,registros,empresas,fechas,anios
0,AUD-0001,5,1,5,3
1,AUD-0002,5,1,5,3
2,AUD-0003,5,1,5,3
3,AUD-0004,5,1,5,3
4,AUD-0005,5,2,5,3
5,AUD-0006,5,1,5,3
6,AUD-0007,5,1,5,2
7,AUD-0008,5,1,5,2
8,AUD-0009,5,1,5,3
9,AUD-0010,5,2,5,3


In [14]:
def revisar_interpretacion(pregunta, correcciones=None):

    # Obtener predicción original
    original = interpretar_consulta(pregunta)

    # Crear una copia independiente
    revisada = original.copy()
    revisada["probabilidades"] = original[
        "probabilidades"
    ].copy()

    # Definir los campos permitidos SIEMPRE
    campos_permitidos = {
        "intencion",
        "entidad",
        "categoria",
        "estado",
        "anios"
    }

    # Aplicar correcciones si existen
    if correcciones is not None:

        desconocidos = (
            set(correcciones) - campos_permitidos
        )

        if desconocidos:
            raise ValueError(
                f"Campos no permitidos: {desconocidos}"
            )

        revisada.update(correcciones)

    # Validar años
    if not isinstance(revisada["anios"], list):
        raise ValueError(
            "El campo 'anios' debe ser una lista."
        )

    if not all(
        isinstance(anio, int)
        for anio in revisada["anios"]
    ):
        raise ValueError(
            "Los años deben ser números enteros."
        )

    # Validar intención y entidad
    if revisada["intencion"] not in config["intenciones"]:
        raise ValueError("Intención no reconocida.")

    if revisada["entidad"] not in config["entidades"]:
        raise ValueError("Entidad no reconocida.")

    # Validar categoría y estado
    categorias_validas = (
        set(config["categorias"]) | {"NINGUNA"}
    )

    estados_validos = (
        set(config["estados"]) | {"NINGUNO"}
    )

    if revisada["categoria"] not in categorias_validas:
        raise ValueError("Categoría no reconocida.")

    if revisada["estado"] not in estados_validos:
        raise ValueError("Estado no reconocido.")

    # Mostrar interpretación original
    print("INTERPRETACIÓN ORIGINAL")

    for campo in [
        "intencion",
        "entidad",
        "categoria",
        "estado",
        "anios"
    ]:
        print(f"{campo}: {original[campo]}")

    # Mostrar interpretación revisada
    print("\nINTERPRETACIÓN REVISADA")

    for campo in [
        "intencion",
        "entidad",
        "categoria",
        "estado",
        "anios"
    ]:
        print(f"{campo}: {revisada[campo]}")

    return preparar_consulta(revisada)
    categorias_validas = (
        set(config["categorias"]) | {"NINGUNA"}
    )

    estados_validos = (
        set(config["estados"]) | {"NINGUNO"}
    )

    if revisada["categoria"] not in categorias_validas:
        raise ValueError("Categoría no reconocida.")

    if revisada["estado"] not in estados_validos:
        raise ValueError("Estado no reconocido.")

    print("INTERPRETACIÓN ORIGINAL")
    for campo in campos_permitidos:
        print(f"{campo}: {original[campo]}")

    print("\nINTERPRETACIÓN REVISADA")
    for campo in campos_permitidos:
        print(f"{campo}: {revisada[campo]}")

    return preparar_consulta(revisada)

In [15]:
pregunta_prueba = (
    "¿Cuántos hallazgos sobre extintores "
    "fueron cerrados en 2025?"
)

# Paso 1: Interpretación inicial
consulta_inicial = revisar_interpretacion(
    pregunta_prueba
)

print("\n¿Consulta válida?:", consulta_inicial["valida"])

# Paso 2: Revisión manual del significado
# Para esta prueba asumimos que los hallazgos
# de extintores se agrupan en emergencias.

consulta_corregida = revisar_interpretacion(
    pregunta_prueba,
    correcciones={
        "intencion": "conteo",
        "entidad": "hallazgos",
        "categoria": "Preparacion y respuesta a emergencias",
        "estado": "Cerrado",
        "anios": [2025]
    }
)

# Paso 3: Preparar, pero NO ejecutar todavía
revision = ejecutar_consulta(
    consulta_corregida,
    confirmada=False
)

print("\nEstado de ejecución:")
print(revision["estado"])

print("\nErrores de validación:")
print(consulta_corregida["errores"])

INTERPRETACIÓN ORIGINAL
intencion: conteo
entidad: hallazgos
categoria: Trabajo en alturas
estado: Cerrado
anios: [2025]

INTERPRETACIÓN REVISADA
intencion: conteo
entidad: hallazgos
categoria: Trabajo en alturas
estado: Cerrado
anios: [2025]

¿Consulta válida?: True
INTERPRETACIÓN ORIGINAL
intencion: conteo
entidad: hallazgos
categoria: Trabajo en alturas
estado: Cerrado
anios: [2025]

INTERPRETACIÓN REVISADA
intencion: conteo
entidad: hallazgos
categoria: Preparacion y respuesta a emergencias
estado: Cerrado
anios: [2025]

Estado de ejecución:
pendiente_confirmacion

Errores de validación:
[]


In [16]:
def ejecutar_consulta_segura(
    consulta,
    confirmada=False,
    tema_especifico=False
):
    # Primero validar la interpretación
    if not consulta["valida"]:
        return {
            "estado": "error",
            "mensaje": "La consulta contiene filtros no válidos.",
            "errores": consulta["errores"]
        }

    interpretacion = consulta["interpretacion"]

    # Proteger consultas sobre auditorías
    if interpretacion["entidad"] == "auditorias":
        return {
            "estado": "no_soportado",
            "mensaje": (
                "El dataset contiene identificadores de "
                "auditoría asociados a distintas fechas, "
                "años o empresas. No es posible obtener "
                "un conteo confiable de auditorías."
            )
        }

    # Proteger preguntas sobre conceptos específicos
    # dentro de una categoría general.
    # Este indicador se establece manualmente por ahora.
    if tema_especifico:
        return {
            "estado": "no_soportado",
            "mensaje": (
                "La categoría general no identifica "
                "necesariamente el concepto específico "
                "solicitado. Necesitamos implementar "
                "búsqueda semántica sobre los hallazgos."
            )
        }

    # Solo ejecutar si se confirmó la interpretación
    return ejecutar_consulta(
        consulta,
        confirmada=confirmada
    )


print("Motor de ejecución segura preparado.")

Motor de ejecución segura preparado.


In [17]:
# ESCENARIO 1: Pregunta sobre auditorías
respuesta_auditorias = ejecutar_consulta_segura(
    consulta_preparada,
    confirmada=True
)

print("1. CONSULTA DE AUDITORÍAS")
print(respuesta_auditorias)


# ESCENARIO 2: Pregunta específica sobre extintores
respuesta_extintores = ejecutar_consulta_segura(
    consulta_corregida,
    confirmada=True,
    tema_especifico=True
)

print("\n2. CONSULTA SOBRE EXTINTORES")
print(respuesta_extintores)


# ESCENARIO 3: Conteo general de hallazgos
consulta_hallazgos = preparar_consulta({
    "pregunta": "¿Cuántos hallazgos hubo en 2025?",
    "intencion": "conteo",
    "entidad": "hallazgos",
    "categoria": "NINGUNA",
    "estado": "NINGUNO",
    "anios": [2025]
})

respuesta_hallazgos = ejecutar_consulta_segura(
    consulta_hallazgos,
    confirmada=True
)

print("\n3. CONSULTA DE HALLAZGOS")
print(respuesta_hallazgos)

1. CONSULTA DE AUDITORÍAS
{'estado': 'no_soportado', 'mensaje': 'El dataset contiene identificadores de auditoría asociados a distintas fechas, años o empresas. No es posible obtener un conteo confiable de auditorías.'}

2. CONSULTA SOBRE EXTINTORES
{'estado': 'no_soportado', 'mensaje': 'La categoría general no identifica necesariamente el concepto específico solicitado. Necesitamos implementar búsqueda semántica sobre los hallazgos.'}

3. CONSULTA DE HALLAZGOS
{'estado': 'correcto', 'entidad': 'hallazgos', 'cantidad': 1781, 'filtros': {'categoria': 'NINGUNA', 'estado': 'NINGUNO', 'anios': [2025]}}


In [18]:
# Preparar textos de los hallazgos
textos_hallazgos = (
    datos_sst["hallazgo"]
    .fillna("")
    .astype(str)
    .tolist()
)

# Representación semántica de todos los registros
embeddings_hallazgos = modelo_nlp.encode(
    textos_hallazgos,
    batch_size=32,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Índice semántico creado correctamente.")
print("Registros:", len(textos_hallazgos))
print("Dimensiones:", embeddings_hallazgos.shape)

Batches: 100%|██████████| 157/157 [00:05<00:00, 30.21it/s]

Índice semántico creado correctamente.
Registros: 5000
Dimensiones: (5000, 384)


In [19]:
def buscar_hallazgos_semanticos(
    consulta,
    categoria=None,
    estado=None,
    anio=None,
    top_k=10
):

    # Generar embedding de la consulta
    embedding_consulta = modelo_nlp.encode(
        [consulta],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    # Similitud coseno
    similitudes = (
        embeddings_hallazgos @ embedding_consulta
    )

    # Trabajar sobre todos los registros
    candidatos = datos_sst.copy()

    # Añadir puntuaciones
    candidatos["similitud"] = similitudes

    # Aplicar filtros confirmados
    if categoria is not None:
        candidatos = candidatos[
            candidatos["categoria"] == categoria
        ]

    if estado is not None:
        candidatos = candidatos[
            candidatos["estado"] == estado
        ]

    if anio is not None:
        candidatos = candidatos[
            candidatos["anio"] == anio
        ]

    # Ordenar según similitud
    resultados = (
        candidatos
        .sort_values(
            "similitud",
            ascending=False
        )
        .head(top_k)
    )

    return resultados[
        [
            "id_hallazgo",
            "fecha_auditoria",
            "categoria",
            "estado",
            "hallazgo",
            "similitud"
        ]
    ].reset_index(drop=True)


print("Buscador semántico preparado.")

Buscador semántico preparado.


In [20]:
resultados_extintores = buscar_hallazgos_semanticos(
    consulta=(
        "Extintores y equipos para combatir incendios "
        "que presentan problemas de mantenimiento, "
        "disponibilidad o inspección"
    ),
    categoria="Preparacion y respuesta a emergencias",
    estado="Cerrado",
    anio=2025,
    top_k=10
)

print("RESULTADOS SEMÁNTICOS SOBRE EXTINTORES")
print("Registros recuperados:", len(resultados_extintores))

display(resultados_extintores)

RESULTADOS SEMÁNTICOS SOBRE EXTINTORES
Registros recuperados: 10


,id_hallazgo,fecha_auditoria,categoria,estado,hallazgo,similitud
0,SST-01256,2025-06-20,Preparacion y respuesta a emergencias,Cerrado,En el registro de control de el taller de mant...,0.556759
1,SST-05000,2025-06-08,Preparacion y respuesta a emergencias,Cerrado,"Segun la inspeccion realizada, un extintor est...",0.538582
2,SST-00968,2025-06-19,Preparacion y respuesta a emergencias,Cerrado,En la verificacion realizada en el area de ser...,0.472191
3,SST-04616,2025-11-04,Preparacion y respuesta a emergencias,Cerrado,Al revisar las condiciones de la planta de pro...,0.449887
4,SST-03896,2025-11-28,Preparacion y respuesta a emergencias,Cerrado,"Segun la inspeccion realizada, el botiquin se ...",0.442079
5,SST-02024,2025-02-09,Preparacion y respuesta a emergencias,Cerrado,Al contrastar la practica con el control previ...,0.414039
6,SST-00056,2025-11-23,Preparacion y respuesta a emergencias,Cerrado,Al revisar las condiciones de el patio operati...,0.407939
7,SST-00452,2025-08-22,Preparacion y respuesta a emergencias,Cerrado,"Segun la inspeccion realizada, la alarma de em...",0.404532
8,SST-02216,2025-02-28,Preparacion y respuesta a emergencias,Cerrado,La muestra revisada revelo que un extintor est...,0.398949
9,SST-03908,2025-02-16,Preparacion y respuesta a emergencias,Cerrado,"Segun la inspeccion realizada, la alarma de em...",0.398225


In [21]:
print("REVISIÓN DE LOS 10 RESULTADOS SEMÁNTICOS")
print("=" * 70)

for i, fila in resultados_extintores.iterrows():

    print(f"\nRESULTADO {i + 1}")
    print("ID:", fila["id_hallazgo"])
    print("Similitud:", round(fila["similitud"], 4))
    print("Categoría:", fila["categoria"])
    print("Estado:", fila["estado"])

    print("\nHallazgo:")
    print(fila["hallazgo"])

    print("-" * 70)

REVISIÓN DE LOS 10 RESULTADOS SEMÁNTICOS

RESULTADO 1
ID: SST-01256
Similitud: 0.5568
Categoría: Preparacion y respuesta a emergencias
Estado: Cerrado

Hallazgo:
En el registro de control de el taller de mantenimiento quedo consignado que un extintor esta obstruido por cajas.
----------------------------------------------------------------------

RESULTADO 2
ID: SST-05000
Similitud: 0.5386
Categoría: Preparacion y respuesta a emergencias
Estado: Cerrado

Hallazgo:
Segun la inspeccion realizada, un extintor esta obstruido por cajas en la obra de adecuacion, lo cual requiere evaluacion.
----------------------------------------------------------------------

RESULTADO 3
ID: SST-00968
Similitud: 0.4722
Categoría: Preparacion y respuesta a emergencias
Estado: Cerrado

Hallazgo:
En la verificacion realizada en el area de servicios generales se constato que un extintor esta obstruido por cajas.
----------------------------------------------------------------------

RESULTADO 4
ID: SST-04616
S

In [22]:
from pathlib import Path

# Crear tabla de evaluación
evaluacion_extintores = resultados_extintores[
    ["id_hallazgo", "hallazgo", "similitud"]
].copy()

evaluacion_extintores["relevante"] = pd.NA

# Carpeta para las evaluaciones
ruta_evaluaciones = Path("evaluaciones")
ruta_evaluaciones.mkdir(exist_ok=True)

archivo_evaluacion = (
    ruta_evaluaciones / "evaluacion_extintores.csv"
)

evaluacion_extintores.to_csv(
    archivo_evaluacion,
    index=False,
    encoding="utf-8-sig"
)

print("Archivo de evaluación creado:")
print(archivo_evaluacion.resolve())

display(evaluacion_extintores)

Archivo de evaluación creado:
D:\VIVIAN\Uis\Quinto semestre\Deep Learning\Proyecto Mini-Transforms\notebooks\evaluaciones\evaluacion_extintores.csv


,id_hallazgo,hallazgo,similitud,relevante
0,SST-01256,En el registro de control de el taller de mant...,0.556759,<NA>
1,SST-05000,"Segun la inspeccion realizada, un extintor est...",0.538582,<NA>
2,SST-00968,En la verificacion realizada en el area de ser...,0.472191,<NA>
3,SST-04616,Al revisar las condiciones de la planta de pro...,0.449887,<NA>
4,SST-03896,"Segun la inspeccion realizada, el botiquin se ...",0.442079,<NA>
5,SST-02024,Al contrastar la practica con el control previ...,0.414039,<NA>
6,SST-00056,Al revisar las condiciones de el patio operati...,0.407939,<NA>
7,SST-00452,"Segun la inspeccion realizada, la alarma de em...",0.404532,<NA>
8,SST-02216,La muestra revisada revelo que un extintor est...,0.398949,<NA>
9,SST-03908,"Segun la inspeccion realizada, la alarma de em...",0.398225,<NA>


In [23]:
def evaluar_precision_busqueda(tabla):

    if tabla["relevante"].isna().any():
        print(
            "Todavía existen resultados sin revisar. "
            "Completa las etiquetas antes de evaluar."
        )
        return None

    etiquetas = pd.to_numeric(
        tabla["relevante"],
        errors="coerce"
    )

    if etiquetas.isna().any() or not etiquetas.isin([0, 1]).all():
        raise ValueError(
            "Las etiquetas deben ser 0 o 1."
        )

    precision = etiquetas.mean()

    print("EVALUACIÓN DE RECUPERACIÓN SEMÁNTICA")
    print("Resultados revisados:", len(tabla))
    print("Resultados relevantes:", int(etiquetas.sum()))
    print(f"Precision@{len(tabla)}: {precision:.2%}")

    return precision


# Leer la tabla de evaluación
evaluacion_guardada = pd.read_csv(archivo_evaluacion)

precision = evaluar_precision_busqueda(
    evaluacion_guardada
)

Todavía existen resultados sin revisar. Completa las etiquetas antes de evaluar.


In [24]:
# Etiquetas revisadas a partir de los textos completos
etiquetas_relevancia = {
    "SST-01256": 1,
    "SST-05000": 1,
    "SST-00968": 1,
    "SST-04616": 1,
    "SST-03896": 0,
    "SST-02024": 1,
    "SST-00056": 0,
    "SST-00452": 0,
    "SST-02216": 1,
    "SST-03908": 0
}

# Trabajar sobre los resultados recuperados
evaluacion_extintores = resultados_extintores[
    ["id_hallazgo", "hallazgo", "similitud"]
].copy()

# Verificar que estemos evaluando los mismos registros
ids_recuperados = set(evaluacion_extintores["id_hallazgo"])

assert ids_recuperados == set(etiquetas_relevancia), (
    "Los resultados han cambiado. "
    "Revisa nuevamente los textos antes de etiquetar."
)

evaluacion_extintores["relevante"] = (
    evaluacion_extintores["id_hallazgo"]
    .map(etiquetas_relevancia)
)

# Guardar etiquetas revisadas
evaluacion_extintores.to_csv(
    archivo_evaluacion,
    index=False,
    encoding="utf-8-sig"
)

print("Evaluación guardada correctamente.")
print("Relevantes:", evaluacion_extintores["relevante"].sum())
print("No relevantes:", (evaluacion_extintores["relevante"] == 0).sum())

display(evaluacion_extintores)

Evaluación guardada correctamente.
Relevantes: 6
No relevantes: 4


,id_hallazgo,hallazgo,similitud,relevante
0,SST-01256,En el registro de control de el taller de mant...,0.556759,1
1,SST-05000,"Segun la inspeccion realizada, un extintor est...",0.538582,1
2,SST-00968,En la verificacion realizada en el area de ser...,0.472191,1
3,SST-04616,Al revisar las condiciones de la planta de pro...,0.449887,1
4,SST-03896,"Segun la inspeccion realizada, el botiquin se ...",0.442079,0
5,SST-02024,Al contrastar la practica con el control previ...,0.414039,1
6,SST-00056,Al revisar las condiciones de el patio operati...,0.407939,0
7,SST-00452,"Segun la inspeccion realizada, la alarma de em...",0.404532,0
8,SST-02216,La muestra revisada revelo que un extintor est...,0.398949,1
9,SST-03908,"Segun la inspeccion realizada, la alarma de em...",0.398225,0


In [25]:
# Leer etiquetas guardadas
evaluacion_guardada = pd.read_csv(archivo_evaluacion)

# Recuperar función anterior
precision_10 = evaluar_precision_busqueda(
    evaluacion_guardada
)

# Evaluar los primeros cinco resultados
primeros_cinco = evaluacion_guardada.head(5)

precision_5 = primeros_cinco["relevante"].mean()

# Comparación
resumen_precision = pd.DataFrame({
    "metrica": ["Precision@5", "Precision@10"],
    "precision": [precision_5, precision_10],
    "relevantes": [
        int(primeros_cinco["relevante"].sum()),
        int(evaluacion_guardada["relevante"].sum())
    ]
})

print("\nRESULTADOS DEL BUSCADOR SEMÁNTICO")
display(resumen_precision)

EVALUACIÓN DE RECUPERACIÓN SEMÁNTICA
Resultados revisados: 10
Resultados relevantes: 6
Precision@10: 60.00%

RESULTADOS DEL BUSCADOR SEMÁNTICO


,metrica,precision,relevantes
0,Precision@5,0.8,4
1,Precision@10,0.6,6


In [26]:
consultas_comparacion = [
    "Extintores obstruidos por cajas u objetos.",

    "Equipos de extinción de incendios con problemas "
    "de acceso, disponibilidad o ubicación.",

    "Problemas de seguridad asociados a extintores."
]

resultados_comparacion = []

for numero, consulta in enumerate(consultas_comparacion, 1):

    resultados = buscar_hallazgos_semanticos(
        consulta=consulta,
        categoria="Preparacion y respuesta a emergencias",
        estado="Cerrado",
        anio=2025,
        top_k=10
    )

    for posicion, fila in resultados.iterrows():

        resultados_comparacion.append({
            "consulta": numero,
            "posicion": posicion + 1,
            "id_hallazgo": fila["id_hallazgo"],
            "similitud": fila["similitud"],
            "hallazgo": fila["hallazgo"]
        })

comparacion_busquedas = pd.DataFrame(
    resultados_comparacion
)

for numero, consulta in enumerate(consultas_comparacion, 1):

    print(f"\nCONSULTA {numero}: {consulta}")

    display(
        comparacion_busquedas.loc[
            comparacion_busquedas["consulta"] == numero,
            ["posicion", "id_hallazgo", "similitud", "hallazgo"]
        ]
    )


CONSULTA 1: Extintores obstruidos por cajas u objetos.


,posicion,id_hallazgo,similitud,hallazgo
0,1,SST-01736,0.741106,Se identifico una desviacion durante la visita...
1,2,SST-00968,0.736760,En la verificacion realizada en el area de ser...
2,3,SST-02024,0.720755,Al contrastar la practica con el control previ...
3,4,SST-02312,0.718686,Al revisar las condiciones de el centro de dis...
4,5,SST-04616,0.713962,Al revisar las condiciones de la planta de pro...
5,6,SST-02216,0.662990,La muestra revisada revelo que un extintor est...
6,7,SST-01256,0.658646,En el registro de control de el taller de mant...
7,8,SST-05000,0.574767,"Segun la inspeccion realizada, un extintor est..."
8,9,SST-04184,0.384592,Durante el recorrido se verifico que el botiqu...
9,10,SST-04472,0.311937,En la verificacion realizada en el patio opera...



CONSULTA 2: Equipos de extinción de incendios con problemas de acceso, disponibilidad o ubicación.


,posicion,id_hallazgo,similitud,hallazgo
10,1,SST-01256,0.422298,En el registro de control de el taller de mant...
11,2,SST-00968,0.405616,En la verificacion realizada en el area de ser...
12,3,SST-01736,0.381795,Se identifico una desviacion durante la visita...
13,4,SST-05000,0.375907,"Segun la inspeccion realizada, un extintor est..."
14,5,SST-00032,0.362929,Al revisar las condiciones de el taller de man...
15,6,SST-02024,0.349237,Al contrastar la practica con el control previ...
16,7,SST-04196,0.333094,Durante el recorrido se verifico que la alarma...
17,8,SST-04616,0.329026,Al revisar las condiciones de la planta de pro...
18,9,SST-02216,0.328980,La muestra revisada revelo que un extintor est...
19,10,SST-03620,0.320493,En el registro de control de el patio operativ...



CONSULTA 3: Problemas de seguridad asociados a extintores.


,posicion,id_hallazgo,similitud,hallazgo
20,1,SST-02024,0.642913,Al contrastar la practica con el control previ...
21,2,SST-02312,0.637580,Al revisar las condiciones de el centro de dis...
22,3,SST-00968,0.635851,En la verificacion realizada en el area de ser...
23,4,SST-02216,0.633891,La muestra revisada revelo que un extintor est...
24,5,SST-01256,0.627784,En el registro de control de el taller de mant...
25,6,SST-05000,0.622606,"Segun la inspeccion realizada, un extintor est..."
26,7,SST-01736,0.615902,Se identifico una desviacion durante la visita...
27,8,SST-04616,0.586345,Al revisar las condiciones de la planta de pro...
28,9,SST-03908,0.379012,"Segun la inspeccion realizada, la alarma de em..."
29,10,SST-00452,0.361186,"Segun la inspeccion realizada, la alarma de em..."


In [27]:
# Tabla con los 30 resultados recuperados
evaluacion_comparativa = comparacion_busquedas[
    [
        "consulta",
        "posicion",
        "id_hallazgo",
        "similitud",
        "hallazgo"
    ]
].copy()

# Incorporar el texto de cada consulta
mapa_consultas = {
    i: texto
    for i, texto in enumerate(consultas_comparacion, 1)
}

evaluacion_comparativa["texto_consulta"] = (
    evaluacion_comparativa["consulta"]
    .map(mapa_consultas)
)

# La relevancia se revisará manualmente
evaluacion_comparativa["relevante"] = pd.NA

ruta_evaluacion_comparativa = (
    ruta_evaluaciones / "evaluacion_tres_consultas.csv"
)

evaluacion_comparativa.to_csv(
    ruta_evaluacion_comparativa,
    index=False,
    encoding="utf-8-sig"
)

print("Archivo creado:")
print(ruta_evaluacion_comparativa.resolve())

print("Resultados por consulta:")
display(
    evaluacion_comparativa.groupby("consulta").size()
)

Archivo creado:
D:\VIVIAN\Uis\Quinto semestre\Deep Learning\Proyecto Mini-Transforms\notebooks\evaluaciones\evaluacion_tres_consultas.csv
Resultados por consulta:


consulta
1    10
2    10
3    10
dtype: int64

In [28]:
for numero, consulta in enumerate(consultas_comparacion, 1):

    print("\n" + "=" * 75)
    print(f"CONSULTA {numero}: {consulta}")
    print("=" * 75)

    seleccion = evaluacion_comparativa[
        evaluacion_comparativa["consulta"] == numero
    ]

    for _, fila in seleccion.iterrows():

        print(f"\nPosición: {fila['posicion']}")
        print("ID:", fila["id_hallazgo"])
        print("Similitud:", round(fila["similitud"], 4))
        print("Hallazgo:", fila["hallazgo"])
        print("-" * 75)


CONSULTA 1: Extintores obstruidos por cajas u objetos.

Posición: 1
ID: SST-01736
Similitud: 0.7411
Hallazgo: Se identifico una desviacion durante la visita: un extintor esta obstruido por cajas.
---------------------------------------------------------------------------

Posición: 2
ID: SST-00968
Similitud: 0.7368
Hallazgo: En la verificacion realizada en el area de servicios generales se constato que un extintor esta obstruido por cajas.
---------------------------------------------------------------------------

Posición: 3
ID: SST-02024
Similitud: 0.7208
Hallazgo: Al contrastar la practica con el control previsto, se encontro que un extintor esta obstruido por cajas.
---------------------------------------------------------------------------

Posición: 4
ID: SST-02312
Similitud: 0.7187
Hallazgo: Al revisar las condiciones de el centro de distribucion, se detecto que un extintor esta obstruido por cajas.
---------------------------------------------------------------------------

P

In [29]:
# Leer evaluación revisada
evaluacion_revisada = pd.read_csv(
    ruta_evaluacion_comparativa
)

# Convertir etiquetas a valores numéricos
evaluacion_revisada["relevante"] = pd.to_numeric(
    evaluacion_revisada["relevante"],
    errors="coerce"
)

resultados_metricas = []

for numero, grupo in evaluacion_revisada.groupby("consulta"):

    grupo = grupo.sort_values("posicion")

    # No calcular métricas con etiquetas pendientes
    if grupo["relevante"].isna().any():
        print(
            f"Consulta {numero}: "
            "tiene resultados pendientes de revisión."
        )
        continue

    if not grupo["relevante"].isin([0, 1]).all():
        print(
            f"Consulta {numero}: etiquetas inválidas."
        )
        continue

    if len(grupo) != 10:
        print(
            f"Consulta {numero}: "
            "se esperaban 10 resultados."
        )
        continue

    resultados_metricas.append({
        "consulta": numero,
        "precision_5": grupo.head(5)["relevante"].mean(),
        "precision_10": grupo.head(10)["relevante"].mean(),
        "relevantes_10": int(grupo["relevante"].sum())
    })

tabla_metricas_busqueda = pd.DataFrame(
    resultados_metricas
)

display(tabla_metricas_busqueda)

Consulta 1: tiene resultados pendientes de revisión.
Consulta 2: tiene resultados pendientes de revisión.
Consulta 3: tiene resultados pendientes de revisión.


""
